<!--nav--> [🗺 Learning path](README.md) · **45/49** · ◀ [Production Hardening](./Production_Hardening_Reliability.ipynb) · [Attention Kernels From Scratch](./Attention_Kernels_From_Scratch.ipynb) ▶

# Anatomy of a Single Decode Step: Where the Milliseconds Actually Go

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Anatomy_Of_A_Decode_Step.ipynb)

Every optimization in this track is really a claim about **one slice of one decode step**. Weight
quantization shrinks the GEMM slice. FP8 KV shrinks the attention slice. CUDA graphs delete the
launch-overhead slice. Speculative decoding removes *whole steps*.

You cannot reason about which optimization to reach for until you know **how big each slice is on
your hardware, at your batch size** — because the slices re-proportion dramatically as those change.
This notebook takes one decode step apart.

| Part | What you'll learn |
|---|---|
| **1** | The six things that happen in every decode step |
| **2** | A quantitative model of each slice, sourced to the notebook that derived it |
| **3** | **The interactive time budget** — watch the slices re-proportion with batch, model and GPU |
| **4** | The overhead cliff: why small batches are dominated by things that aren't math |
| **5** | **Which optimization cuts which slice** — the map that makes the whole track legible |
| **6** | **Measure it yourself**: a portable CUDA/ROCm profiler that times each slice |

**Runs on:** any CPU for Parts 1–5. Part 6 profiles whatever GPU you have (NVIDIA *or* AMD).

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · What actually happens in one decode step

A decode step produces **exactly one token per sequence in the batch**. Here is everything the GPU
does, in order:

```
 ┌─ 1. SCHEDULE ─────────── pick which requests are in this step, allocate KV blocks   (CPU)
 │  2. LAUNCH ───────────── enqueue every kernel for every layer                       (CPU→GPU)
 │  3. GEMMs ────────────── QKV proj, O proj, MLP up/gate/down — reads ALL the weights (GPU)
 │  4. ATTENTION ────────── each sequence attends over its own KV cache                (GPU)
 │  5. COMMUNICATION ────── all-reduce (TP) / all-to-all (MoE), if sharded             (GPU+fabric)
 └─ 6. SAMPLE ──────────── logits → softmax/top-p → one token id per sequence          (GPU)
```

Three of these scale with **batch size**, two scale with **context length**, and one is
**constant per step no matter what you do**. That asymmetry is the whole story:

| Slice | Scales with | Bound by |
|---|---|---|
| GEMMs | batch (compute), **not** batch (memory) | **bandwidth** at low batch, compute at high |
| Attention | batch × context | **bandwidth** (reads the KV cache) |
| Communication | batch × hidden × TP topology | interconnect |
| Sampling | batch × vocab | bandwidth, usually small |
| Launch overhead | **nothing** — fixed per step | CPU, kernel count |
| Scheduling | number of requests | CPU |

**The critical one is the fixed cost.** At batch 1 it can be *half your step time*; at batch 256 it's
noise. Any benchmark that measures only one batch size is measuring one point on a curve that
changes shape.

## Part 2 · The model

Each slice, with its source:

| Slice | Formula | From |
|---|---|---|
| GEMM (memory-bound) | `params × bytes / bandwidth` | [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb), [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) |
| GEMM (compute-bound) | `2 × params × batch / FLOPS` | [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) |
| Attention | `2 × layers × kv_heads × head_dim × kv_bytes × ctx × batch / bandwidth` | [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) |
| Sampling | `batch × vocab × 4 bytes × passes / bandwidth` | — |
| Communication | `2 × hidden × batch × bytes × 2(N−1)/N × layers / link` | [Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb) |
| Launch overhead | `kernels_per_layer × layers / launch_rate` | measured in Part 6 |

In [ ]:
# One decode step, decomposed. Every term is bytes-moved or FLOPs, divided by a rate.
HW = {
  "T4":        dict(bw=0.32e12, tf=65e12,   link=None,   launch_us=2.2),
  "L4":        dict(bw=0.30e12, tf=121e12,  link=None,   launch_us=1.8),
  "A100 80GB": dict(bw=2.04e12, tf=312e12,  link=300e9,  launch_us=1.5),
  "H100 SXM":  dict(bw=3.35e12, tf=990e12,  link=900e9,  launch_us=1.2),
  "MI300X":    dict(bw=5.30e12, tf=1307e12, link=800e9,  launch_us=1.5),
  "B200":      dict(bw=8.00e12, tf=2250e12, link=1800e9, launch_us=1.0),
}
MODELS = {
  "Qwen2.5-0.5B": dict(params=0.5e9, layers=24, hidden=896,  kv_heads=2, head_dim=64,  vocab=152000),
  "Llama-3.1-8B": dict(params=8e9,   layers=32, hidden=4096, kv_heads=8, head_dim=128, vocab=128000),
  "Llama-3.1-70B":dict(params=70e9,  layers=80, hidden=8192, kv_heads=8, head_dim=128, vocab=128000),
}
KERNELS_PER_LAYER = 14        # QKV, RoPE, attn, O, norms, MLP up/gate/down, residuals, ...

def decode_step(model, gpu, batch=32, ctx=2048, weight_bytes=2.0, kv_bytes=2.0,
                tp=1, graphs=True, bw_eff=0.75, flop_eff=0.60, lora=False):
    m, g = MODELS[model], HW[gpu]
    bw = g["bw"] * bw_eff * tp
    flops = g["tf"] * flop_eff * tp

    # 3. GEMMs: read every weight once per step; do 2*params*batch FLOPs with them.
    gemm_mem = (m["params"] * weight_bytes) / bw
    gemm_cmp = (2 * m["params"] * batch) / flops
    gemm = max(gemm_mem, gemm_cmp)

    # 4. Attention: read each sequence's whole KV cache.
    kv_per_tok = 2 * m["layers"] * m["kv_heads"] * m["head_dim"] * kv_bytes
    attention = (kv_per_tok * ctx * batch) / bw

    # 6. Sampling: logits are [batch, vocab] fp32, read/written a few times.
    sampling = (batch * m["vocab"] * 4 * 3) / bw

    # 5. Communication: two all-reduces per layer under tensor parallelism (distributed-serving).
    comm = 0.0
    if tp > 1 and g["link"]:
        per_layer = 2 * m["hidden"] * batch * 2 * (2 * (tp - 1) / tp)
        comm = per_layer * m["layers"] / g["link"]

    # 2. Launch overhead: fixed per step. CUDA/HIP graphs collapse it to a single replay.
    n_kernels = KERNELS_PER_LAYER * m["layers"] + (2 * m["layers"] if lora else 0)
    launch = (g["launch_us"] * 1e-6) * (3 if graphs else n_kernels)

    parts = {"GEMMs (weights)": gemm, "attention (KV)": attention,
             "communication": comm, "sampling": sampling, "launch overhead": launch}
    total = sum(parts.values())
    return {"parts": parts, "total_s": total, "gemm_bound": "compute" if gemm_cmp > gemm_mem else "memory",
            "tokens_per_s": batch / total, "tpot_ms": total * 1000}

r = decode_step("Llama-3.1-8B", "H100 SXM", batch=32, ctx=2048)
print("Llama-3.1-8B on H100, batch 32, 2k context:\n")
for name, sec in sorted(r["parts"].items(), key=lambda kv: -kv[1]):
    print(f"  {name:<20}{sec*1000:>8.3f} ms  ({sec/r['total_s']:>5.1%})")
print(f"  {'TOTAL':<20}{r['total_s']*1000:>8.3f} ms   -> TPOT {r['tpot_ms']:.2f} ms, "
      f"{r['tokens_per_s']:,.0f} tok/s")
print(f"\n  GEMM slice is {r['gemm_bound']}-bound at this batch size.")

## Part 3 · The interactive time budget

Now the useful part: change the batch size, model, GPU and context, and watch the **shape** of the
step change. This is the picture to keep in your head when someone proposes an optimization.

In [ ]:
# Precompute the budget across the grid so the browser can explore it instantly.
grid = []
for gpu in HW:
    for model in MODELS:
        for batch in (1, 2, 4, 8, 16, 32, 64, 128, 256):
            for ctx in (512, 2048, 8192, 32768):
                for graphs in (True, False):
                    r = decode_step(model, gpu, batch=batch, ctx=ctx, graphs=graphs)
                    grid.append({"gpu": gpu, "model": model, "batch": batch, "ctx": ctx,
                                 "graphs": graphs, "total_ms": round(r["total_s"]*1000, 4),
                                 "tpot_ms": round(r["tpot_ms"], 3),
                                 "tps": round(r["tokens_per_s"], 1),
                                 "bound": r["gemm_bound"],
                                 **{k: round(v*1000, 5) for k, v in r["parts"].items()}})
print(f"precomputed {len(grid):,} decode-step budgets")

JS = r'''
const SLICES = ["GEMMs (weights)","attention (KV)","communication","sampling","launch overhead"];
const COLORS = {"GEMMs (weights)":"#1976d2","attention (KV)":"#e53935","communication":"#8e24aa",
                "sampling":"#43a047","launch overhead":"#fb8c00"};
const ctr = root.append("div").style("font","13px system-ui").style("margin-bottom","8px");
function dd(label, vals, def) {
  const w = ctr.append("label").style("margin-right","14px");
  w.append("span").text(label+" ");
  const s = w.append("select").style("font-size","13px");
  s.selectAll("o").data(vals).join("option").attr("value",d=>d).text(d=>d);
  if (def !== undefined) s.property("value", def);
  s.on("change", draw);
  return s;
}
const uniq = k => [...new Set(data.map(d=>d[k]))];
const gpuS = dd("GPU:", uniq("gpu"), "H100 SXM");
const modS = dd("model:", uniq("model"), "Llama-3.1-8B");
const ctxS = dd("context:", uniq("ctx").sort((a,b)=>a-b), 2048);
const grS  = dd("graph capture:", ["on","off"], "on");

const M = {top: 22, right: 150, bottom: 44, left: 62};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom - 34;
const svg = root.append("svg").attr("width",W).attr("height",H).append("g")
    .attr("transform",`translate(${M.left},${M.top})`);
const x = d3.scaleBand().range([0,iw]).padding(0.18);
const y = d3.scaleLinear().range([ih,0]);
const xAxis = svg.append("g").attr("transform",`translate(0,${ih})`);
const yAxis = svg.append("g");
svg.append("text").attr("x",iw/2).attr("y",ih+36).attr("text-anchor","middle")
   .style("font-size","12px").text("batch size");
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-46)
   .attr("text-anchor","middle").style("font-size","12px").text("share of one decode step");
const note = root.append("div").style("font","12px ui-monospace,monospace").style("color","#455a64");

SLICES.forEach((s,i) => {
  svg.append("rect").attr("x",iw+12).attr("y",i*18).attr("width",11).attr("height",11)
     .attr("rx",2).attr("fill",COLORS[s]);
  svg.append("text").attr("x",iw+28).attr("y",i*18+10).style("font-size","10.5px").text(s);
});

function draw() {
  const gpu=gpuS.property("value"), mod=modS.property("value");
  const ctx=+ctxS.property("value"), graphs=grS.property("value")==="on";
  const rows = data.filter(d=>d.gpu===gpu&&d.model===mod&&d.ctx===ctx&&d.graphs===graphs)
                   .sort((a,b)=>a.batch-b.batch);
  x.domain(rows.map(d=>d.batch));
  y.domain([0,1]);
  xAxis.call(d3.axisBottom(x));
  yAxis.call(d3.axisLeft(y).ticks(5).tickFormat(d3.format(".0%")));
  const stack = d3.stack().keys(SLICES).value((d,k)=>d[k]/d.total_ms);
  const series = stack(rows);
  const g = svg.selectAll("layer").data(series, d=>d.key);
  g.join("g").attr("fill",d=>COLORS[d.key])
   .selectAll("rect").data(d=>d.map(v=>({...v, key:d.key}))).join("rect")
   .attr("x",d=>x(d.data.batch)).attr("width",x.bandwidth())
   .attr("y",d=>y(d[1])).attr("height",d=>Math.max(0,y(d[0])-y(d[1])))
   .select(function(){return this;});
  svg.selectAll("g").selectAll("rect").selectAll("title").remove();
  svg.selectAll("g").selectAll("rect").append("title")
     .text(d => d.key ? `${d.key}: ${d3.format(".1%")(d[1]-d[0])} of the step` : "");
  const b1 = rows.find(r=>r.batch===1), b32 = rows.find(r=>r.batch===32) || rows[rows.length-1];
  note.text(
`batch   1 : step ${b1.total_ms.toFixed(3)} ms · TPOT ${b1.tpot_ms.toFixed(2)} ms · ${d3.format(".0f")(b1.tps)} tok/s · GEMM ${b1.bound}-bound
batch ${String(b32.batch).padStart(3)} : step ${b32.total_ms.toFixed(3)} ms · TPOT ${b32.tpot_ms.toFixed(2)} ms · ${d3.format(".0f")(b32.tps)} tok/s · GEMM ${b32.bound}-bound
biggest slice at batch 1: ${d3.greatest(["GEMMs (weights)","attention (KV)","sampling","launch overhead"], k=>b1[k])}`);
}
draw();
'''
show_d3(JS, grid, height=380)

**Three things to try, in order — each one is a lesson:**

1. **Set graph capture to `off`.** At batch 1 the orange launch-overhead slice explodes to dominate
   the step. This is *why* CUDA/HIP graphs exist, and why vLLM's V1 engine captures the decode step
   ([vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb)). Nothing about the math changed; you were just paying to *ask* for it.
2. **Slide the context to 32768.** The red attention slice takes over, and the GEMM slice — the one
   everybody optimizes — becomes a minority of the step. At long context, **quantizing weights
   barely helps; quantizing the KV cache does** ([Long-Context Serving](./LongContext_KV_Compression_Serving.ipynb)).
3. **Compare a T4 to a B200 at batch 1.** The *proportions* barely move even though the absolute time
   does: both are bandwidth-dominated. Now compare at batch 256 — the fast GPU flips to compute-bound
   and the shape changes. Same code, different bottleneck ([The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb)).

## Part 4 · The overhead cliff

The fixed per-step cost deserves its own look, because it produces the most counter-intuitive
behavior in serving: **at small batch, a big chunk of your step isn't math at all.**

In [ ]:
print("Share of the decode step that is NOT arithmetic (launch overhead), Llama-3.1-8B:\n")
print(f"{'batch':>7}{'graphs ON':>26}{'graphs OFF':>26}")
print(f"{'':>7}{'step ms':>12}{'overhead':>14}{'step ms':>12}{'overhead':>14}")
print("-" * 60)
for b in (1, 2, 4, 8, 16, 32, 64, 128):
    on = decode_step("Llama-3.1-8B", "H100 SXM", batch=b, ctx=2048, graphs=True)
    off = decode_step("Llama-3.1-8B", "H100 SXM", batch=b, ctx=2048, graphs=False)
    print(f"{b:>7}{on['total_s']*1000:>12.3f}{on['parts']['launch overhead']/on['total_s']:>13.1%}"
          f"{off['total_s']*1000:>12.3f}{off['parts']['launch overhead']/off['total_s']:>13.1%}")

print("\nWithout graph capture, a batch-1 step is mostly the CPU asking the GPU to do things.")
print("A 32-layer model launches ~450 kernels per step; at ~1.2us each that is ~0.5 ms of pure")
print("bookkeeping - comparable to the weight read itself on a fast GPU.")
print("\nThis is also why the same model feels slower in plain PyTorch than in vLLM (vLLM, quantization):")
print("eager execution pays this cost on every single token.")
print("\nAnd it is why MoE (MoE) and multi-LoRA (multi-LoRA) need care: both ADD kernels per step,")
print("so both eat into exactly this slice.")

## Part 5 · Which optimization cuts which slice

This is the map that makes the entire track legible. Every technique in the serving track removes
time from **one specific slice** — and therefore only helps when that slice is big:

| Optimization | Cuts | Helps when | Does nothing when | Notebook |
|---|---|---|---|---|
| **Weight quantization (int4/FP8)** | GEMM (memory side) | GEMM slice is large & memory-bound | you're compute-bound, or attention dominates | [Quantized Serving Showdown](./Quantized_Serving_Showdown.ipynb) |
| **FP8/int4 KV cache** | attention | long context, big batch | short context | [Long-Context Serving](./LongContext_KV_Compression_Serving.ipynb) |
| **KV eviction / SWA** | attention | very long context | short context | [Long-Context Serving](./LongContext_KV_Compression_Serving.ipynb) |
| **Larger batch** | GEMM *per token* (amortizes it) | memory-bound GEMM | already compute-bound | [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) |
| **CUDA/HIP graphs** | launch overhead | small batch | large batch | [vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) |
| **Speculative decoding** | *whole steps* | spare compute exists (low/mid batch) | compute-saturated | [Speculative Decoding](./Speculative_Decoding_Advanced_Serving.ipynb) |
| **Prefix caching** | *prefill*, not decode | repeated prefixes | one-shot prompts | [vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) |
| **Tensor parallelism** | GEMM + attention (splits both) | model doesn't fit; latency-critical | you're already fast enough | [Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb) |
| **Better kernels (Marlin/AITER)** | GEMM (compute side) | compute-bound | memory-bound | [Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb) |

**Read the "does nothing when" column carefully.** It's the reason two teams get wildly different
results from the same optimization: they were standing in different parts of this budget.

Let's make that concrete — the same optimization, applied at two batch sizes:

In [ ]:
def apply_opt(base_kwargs, **override):
    a = decode_step(**base_kwargs)
    b = decode_step(**{**base_kwargs, **override})
    return a["tokens_per_s"], b["tokens_per_s"], b["tokens_per_s"] / a["tokens_per_s"]

SCENARIOS = [
    ("interactive (batch 1, 2k ctx)",  dict(model="Llama-3.1-8B", gpu="H100 SXM", batch=1, ctx=2048)),
    ("serving    (batch 64, 2k ctx)",  dict(model="Llama-3.1-8B", gpu="H100 SXM", batch=64, ctx=2048)),
    ("long-ctx   (batch 8, 32k ctx)",  dict(model="Llama-3.1-8B", gpu="H100 SXM", batch=8, ctx=32768)),
]
OPTS = [("int4 weights", dict(weight_bytes=0.5)),
        ("fp8 KV cache", dict(kv_bytes=1.0)),
        ("graph capture", dict(graphs=True)),
        ("no graph capture", dict(graphs=False))]

for label, base in SCENARIOS:
    print(f"\n{label}")
    print("  " + "-" * 62)
    for opt_name, override in OPTS:
        if opt_name == "graph capture":
            before, after, gain = apply_opt({**base, "graphs": False}, graphs=True)
        else:
            before, after, gain = apply_opt(base, **override)
        verdict = ("big win" if gain > 1.5 else "helps" if gain > 1.08
                   else "negligible" if gain > 0.95 else "HURTS")
        print(f"  {opt_name:<20}{before:>9,.0f} -> {after:>9,.0f} tok/s   {gain:>5.2f}x   {verdict}")

print("\n(The int4 numbers here are the pure bandwidth argument: 4x fewer weight bytes.")
print(" Real int4 lands below this because dequantization costs compute and kernel quality")
print(" varies by vendor - quantization measured it, and optimization-stack models the penalty.)")
print("\n\nSame model, same GPU, same four optimizations - completely different rankings.")
print("int4 weights is transformative at batch 1 and mediocre at 32k context, where the")
print("attention slice dominates and fp8 KV is the one that matters instead.")
print("\nThis is why 'best practices' lists are useless without a workload attached, and why")
print("optimization-stack studies how these COMBINE rather than treating them as a menu.")

## Part 6 · Measure your own step

The model above is arithmetic. Here's the measurement — and, as in [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb), it runs on
**either vendor** because it only uses PyTorch primitives.

We time each slice by constructing it in isolation at realistic shapes:

- **GEMM slice**: the per-layer projections at `[batch, hidden] × [hidden, hidden]`
- **Attention slice**: reading a KV cache of `[batch, ctx, kv_heads, head_dim]`
- **Sampling slice**: a `[batch, vocab]` softmax + top-p style pass
- **Launch overhead**: many tiny kernels, to expose the fixed cost directly

In [ ]:
# Portable micro-profiler: NVIDIA (CUDA) or AMD (ROCm). No vendor-specific code.
# GPU-gated: degrade gracefully when PyTorch is absent, not just when the GPU is.
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    torch = None
    HAS_GPU = False
    print("PyTorch is not installed here - skipping the GPU section.")
import time

def bench(fn, iters=50, warmup=10):
    for _ in range(warmup): fn()
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(iters): fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1000        # ms

if not HAS_GPU:
    print("No GPU detected - Parts 1-5 gave you the full model.")
    print("Run this cell on any CUDA or ROCm box to measure YOUR decode step.")
else:
    hip = getattr(torch.version, "hip", None)
    props = torch.cuda.get_device_properties(0)
    print(f"{'AMD (ROCm ' + hip + ')' if hip else 'NVIDIA (CUDA ' + str(torch.version.cuda) + ')'}"
          f" · {props.name} · {props.total_memory/1e9:.0f} GB\n")

    BATCH, HIDDEN, LAYERS, CTX = 32, 4096, 32, 2048
    KVH, HD, VOCAB = 8, 128, 128000
    dt = torch.float16
    dev = "cuda"

    # --- GEMM slice: the projections one layer performs, times layer count ---
    x = torch.randn(BATCH, HIDDEN, device=dev, dtype=dt)
    w_qkv = torch.randn(HIDDEN, HIDDEN + 2 * KVH * HD, device=dev, dtype=dt)
    w_o   = torch.randn(HIDDEN, HIDDEN, device=dev, dtype=dt)
    w_up  = torch.randn(HIDDEN, 4 * HIDDEN, device=dev, dtype=dt)
    w_dn  = torch.randn(4 * HIDDEN, HIDDEN, device=dev, dtype=dt)
    def one_layer_gemms():
        h = x @ w_qkv
        y = x @ w_o
        u = x @ w_up
        return u @ w_dn
    gemm_ms = bench(one_layer_gemms) * LAYERS

    # --- Attention slice: read the KV cache for every sequence ---
    k = torch.randn(BATCH, CTX, KVH, HD, device=dev, dtype=dt)
    v = torch.randn(BATCH, CTX, KVH, HD, device=dev, dtype=dt)
    q = torch.randn(BATCH, 1, KVH, HD, device=dev, dtype=dt)
    def one_layer_attention():
        scores = torch.einsum("bqhd,bkhd->bhqk", q, k).float().softmax(-1).to(dt)
        return torch.einsum("bhqk,bkhd->bqhd", scores, v)
    attn_ms = bench(one_layer_attention, iters=30) * LAYERS

    # --- Sampling slice ---
    logits = torch.randn(BATCH, VOCAB, device=dev, dtype=torch.float32)
    def sample():
        p = logits.softmax(-1)
        top = p.topk(50, dim=-1)
        return top.indices[:, 0]
    sample_ms = bench(sample)

    # --- Launch overhead: many trivial kernels ---
    tiny = torch.zeros(8, device=dev)
    n_kernels = 14 * LAYERS
    def many_launches():
        for _ in range(n_kernels): tiny.add_(1.0)
    launch_ms = bench(many_launches, iters=20)

    total = gemm_ms + attn_ms + sample_ms + launch_ms
    print(f"Measured decode step (batch {BATCH}, {LAYERS} layers, {CTX} ctx, fp16):\n")
    print(f"{'slice':<24}{'ms':>9}{'share':>9}")
    print("-" * 42)
    for name, ms in (("GEMMs (weights)", gemm_ms), ("attention (KV)", attn_ms),
                     ("sampling", sample_ms), ("launch overhead", launch_ms)):
        print(f"{name:<24}{ms:>9.3f}{ms/total:>9.1%}")
    print(f"{'TOTAL':<24}{total:>9.3f}")
    print(f"\nimplied TPOT {total:.2f} ms · {BATCH/(total/1000):,.0f} tok/s aggregate")

    print("\nCompare with the model from Part 2:")
    pred = decode_step("Llama-3.1-8B", "H100 SXM", batch=BATCH, ctx=CTX)
    for name in ("GEMMs (weights)", "attention (KV)", "sampling", "launch overhead"):
        print(f"  {name:<24}model {pred['parts'][name]*1000:>7.3f} ms")
    print("\nDifferences are expected and informative: the model assumes ideal kernels and")
    print("your GPU is not the one hardcoded above. Edit HW[] with your measured bandwidth")
    print("and TFLOP/s from roofline Part 7, then re-run Part 3.")

### What to do with your measured breakdown

1. **Find your biggest slice.** That is the only slice worth optimizing, and Part 5's table tells you
   which techniques touch it.
2. **Check the overhead slice.** If it's more than ~10% of the step, graph capture is either off or
   ineffective — that's a config fix, not an engineering project.
3. **Re-measure at your real batch size and context.** The shape changes so much that a breakdown
   taken at the wrong operating point will point you at the wrong optimization.
4. **Then read [The Optimization Stack](./The_Optimization_Stack.ipynb)** — because once you fix the biggest
   slice, the *second* biggest becomes the bottleneck, and the optimizations interact.

## Recap

- A decode step is **six things**: schedule, launch, GEMMs, attention, communication, sampling.
- They scale differently: GEMM with batch (compute side only), attention with batch × context,
  overhead with **nothing**.
- **At batch 1 without graph capture, a large share of the step is not arithmetic.**
- **At long context, attention — not the weights — dominates**, which flips which quantization matters.
- **Every optimization in this track cuts exactly one slice.** Match the technique to your biggest
  slice, or you're optimizing something that isn't your problem.

### Further reading
- [Transformer Inference Arithmetic](https://kipp.ly/transformer-inference-arithmetic/) — the same decomposition, done by hand
- [FlashAttention-2](https://arxiv.org/abs/2307.08691) — why the attention slice is smaller than naive math suggests
- [CUDA Graphs](https://developer.nvidia.com/blog/cuda-graphs/) · [HIP Graphs](https://rocm.docs.amd.com/projects/HIP/en/latest/how-to/hipgraph.html) — the launch-overhead fix
- Companion notebooks: [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) (where the formulas come from), [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) (the hardware constants), [The Optimization Stack](./The_Optimization_Stack.ipynb) (how the fixes compose)